# Importing Configs from .env

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

PASS_MARK = int(os.getenv("PASS_MARK"))
LATE_FINE_PER_DAY = int(os.getenv("LATE_FINE_PER_DAY"))

# Custom Exceptions

In [2]:
class StudentError(Exception):
    pass


class ValidationError(StudentError):
    pass


class FileLoadError(StudentError):
    pass

# Student Class

In [3]:
class Student:

    def __init__(self, sid, name, marks, days_late):
        self.sid = sid
        self.name = name
        self.marks = int(marks)
        self.days_late = int(days_late)

    def is_failed(self):
        return self.marks < PASS_MARK

    def __str__(self):
        return f"{self.sid} - {self.name} - {self.marks}"

# Loading Students from CSV file

In [4]:
import csv

def load_students(path):

    students = []

    try:
        with open(path, "r") as f:
            reader = csv.DictReader(f)

            for row in reader:
                student = Student(
                    row["id"],
                    row["name"],
                    row["marks"],
                    row["late_days"]
                )

                students.append(student)

    except Exception as e:
        raise FileLoadError(str(e))

    return students

# Validating Students & Fine Calculation

In [6]:
def validate_students(students):

    for s in students:

        if not s.name:
            raise ValidationError("Missing name")

        if s.marks < 0 or s.marks > 100:
            raise ValidationError("Invalid marks")
        
def calculate_fine(student):

    return student.days_late * LATE_FINE_PER_DAY

# Decorator for Logging

In [7]:
import time

def log_execution(func):

    def wrapper(*args, **kwargs):

        start = time.time()

        result = func(*args, **kwargs)

        end = time.time()

        print(f"{func.__name__} took {end-start:.2f}s")

        return result

    return wrapper

# Async Report Generator

In [8]:
import asyncio
from datetime import datetime

async def generate_report(students):

    filename = f"report_{datetime.now().strftime('%H%M%S')}.txt"

    with open(filename, "w") as f:

        for s in students:

            fine = calculate_fine(s)

            status = "FAILED" if s.is_failed() else "PASS"

            line = f"{s.sid}, {s.name}, {s.marks}, {status}, fine={fine}\n"

            f.write(line)

            await asyncio.sleep(0)

    print("Report generated:", filename)

In [10]:
import asyncio

@log_execution
async def main():

    students = load_students("students.csv")

    validate_students(students)

    await generate_report(students)


await main()

main took 0.00s
Report generated: report_202029.txt
